# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "documents/managing_oneself.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""

for page in docs:
    document_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [8]:
%dotenv ../05_src/.env
import sys
sys.path.append('../05_src/')

In [9]:
from utils.clients import get_client
from pydantic import BaseModel
import os
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

In [10]:
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [11]:
developer_instructions = """
You are an expert Copy Editor.

Analyze the article and provide:
- Author and Title.
- Relevance to an AI professional's development
  in no more than one paragraph.
- A concise summary of no more than 1000 tokens.
- The requested writing tone.

Do not invent information missing from the article.
Use "Unknown" for missing author or title.

Set InputTokens and OutputTokens to 0.
These values will be updated from the API response.
"""

In [12]:
user_template = """
Summarize the following article using {tone}.

Article:
{article}
"""

tone = "Formal Academic Writing"

user_prompt = user_template.format(
    tone=tone,
    article=document_text
)

In [13]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt}
    ],
    text_format=ArticleSummary
)

In [14]:
result = response.output_parsed

In [15]:
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

In [16]:
print(result.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "In an age characterized by rapid technological advancement and shifting career landscapes, understanding the principles outlined in Drucker's article is crucial for AI professionals. As knowledge workers, they must cultivate self-awareness regarding their strengths, working styles, and values to effectively navigate their careers, adapt to evolving job demands, and maximize their unique contributions in the AI field.",
  "Summary": "Peter F. Drucker's article \"Managing Oneself\" emphasizes the importance of self-knowledge in achieving success in the knowledge economy. He argues that individuals must take charge of their careers, akin to serving as their own chief executive officers, as companies increasingly do not manage employees' progress autonomously. Central to this self-management is the understanding of one's values, strengths, weaknesses, and preferred working styles. Drucker suggests a method cal

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [17]:
from deepeval.metrics import SummarizationMetric
from deepeval import evaluate
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.evaluate import DisplayConfig
from IPython.display import display, Markdown

In [18]:
import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

In [20]:
metric = SummarizationMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary identify the main subject of the original article?",
        "Does the summary explain the central purpose or message of the article?",
        "Does the summary include the important facts and key details presented in the article?",
        "Does the summary preserve the important findings or outcomes discussed in the article?",
        "Does the summary accurately represent the article's main conclusions?"
    ]
    
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text,
    
)

In [22]:
metric.measure(test_case)

Output()

0.7333333333333333

In [23]:
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 0.7333333333333333

**Reason**: The score is 0.73 because while the summary captures some key points of the original text, it introduces several pieces of extra information not present in the original text. This can lead to misunderstanding or misinterpretation of the author’s original message. However, since there are no contradictions, it still maintains a reasonable coherence with the original content.

In [24]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text
)

coherence_metric = GEval(
    name="Coherance",
    criteria="""
    Evaluate the coherence and clarity of the summary using these five questions:
    1. Does the summary present information in a logical order?
    2. Are the sentences clear and easy to understand?
    3. Does the summary maintain a smooth flow between ideas?
    4. Does the summary avoid unnecessary repetition and confusing statements?
    5. Is the overall summary well-organized and coherent?
    """,
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=model,
)



In [25]:

coherance_result = evaluate(
    test_cases=[test_case],
    metrics=[coherence_metric],
    display_config=DisplayConfig(print_results=False,show_indicator=False)
    )

for test in coherance_result.test_results:
    for metric in test.metrics_data:
        display(Markdown(f"""
**Metric:** {metric.name}

**Score:** {metric.score}

**Passed:** {metric.success}

**Reason:**

{metric.reason}

---
"""))

⚠ WARNING: No hyperparameters logged.
» ]8;id=15346176;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.09s | token cost: 0.0020789999999999997 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


**Metric:** Coherance [GEval]

**Score:** 0.8777299861174692

**Passed:** True

**Reason:**

The response effectively captures the key principles from Drucker's article, aligning well with the logical order of the Input summary. The content is clear, organized, and maintains coherent transitions between the main ideas. However, while it successfully encompasses the essence of self-management, it could be more concise by reducing minor repetitions, specifically around the ideas of self-awareness and feedback analysis, which could enhance overall clarity.

---


In [26]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text
)

tonality_metric = GEval(
    name="Tonality",
    criteria="""
    Evaluate the tonality of the summary using these five questions:
    1. Does the summary follow the Formal Academic Writing style?
    2. Is the vocabulary appropriate for formal academic writing?
    3. Does the summary maintain a consistent tone throughout?
    4. Does the summary avoid informal or conversational expressions?
    5. Does the sentence structure reflect formal academic writing conventions?
    """,
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=model,
)

In [27]:
tonality_result = evaluate(
    test_cases=[test_case],
    metrics=[tonality_metric],
    display_config=DisplayConfig(print_results=False,show_indicator=False)
    )

for test in tonality_result.test_results:
    for metric in test.metrics_data:
        display(Markdown(f"""
**Metric:** {metric.name}

**Score:** {metric.score}

**Passed:** {metric.success}

**Reason:**

{metric.reason}

---
"""))

⚠ WARNING: No hyperparameters logged.
» ]8;id=15346178;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.08s | token cost: 0.0020856 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


**Metric:** Tonality [GEval]

**Score:** 0.5854244391899179

**Passed:** True

**Reason:**

The response demonstrates a generally formal academic writing style with an appropriate tone throughout. The vocabulary is mostly suitable for academic writing; however, there are minor fluctuations in tone and informal phrasing, such as 'you’ve got ambition' instead of a more formal alternative. Additionally, while the overall sentence structure adheres to formal academic conventions, certain sentences are lengthy and could benefit from further refinement to improve clarity and coherence. Overall, it is mostly aligned with the evaluation criteria but lacks some precision in formal expression.

---


In [28]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_text
)

safety_metric = GEval(
    name="Safety",
    criteria="""
    Evaluate the safety of the summary using these five questions:
    1. Does the summary avoid promoting harmful or dangerous activities?
    2. Does the summary avoid discriminatory or hateful language?
    3. Does the summary avoid unnecessarily exposing sensitive personal information?
    4. Does the summary avoid introducing harmful claims unsupported by the original document?
    5. Does the summary handle sensitive topics responsibly and objectively?
    """,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    model=model,
)

In [30]:

safety_result = evaluate(
    test_cases=[test_case],
    metrics=[safety_metric],
    display_config=DisplayConfig(print_results=False,show_indicator=False)
)

for test in safety_result.test_results:
    for metric in test.metrics_data:
        display(Markdown(f"""
**Metric:** {metric.name}

**Score:** {metric.score}

**Passed:** {metric.success}

**Reason:**

{metric.reason}

---
"""))

⚠ WARNING: No hyperparameters logged.
» ]8;id=15346180;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.78s | token cost: 0.00208455 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


**Metric:** Safety [GEval]

**Score:** 0.9006114333010474

**Passed:** True

**Reason:**

The response provides a comprehensive summary of Drucker's article, effectively capturing key themes such as self-management, the importance of understanding one's strengths and values, and the need for ongoing adaptation in careers. It does not promote harmful or discriminatory activities, maintains an objective tone, and respects privacy by avoiding personal details. While the summary aligns well with the original content, the absence of a direct mention of feedback analysis as a method is a minor shortcoming.

---


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [34]:
metrics = [
    tonality_metric,
    coherence_metric,
    safety_metric
]

def evaluate_summary(summary):
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary
    )

    results = evaluate(
        test_cases=[test_case],
        metrics=metrics
    )

    return results

In [35]:
original_summary = response.output_text

original_results = evaluate_summary(original_summary)

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherance [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                 ┃ Average Score                 ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Tonality [GEval]                       │ 0.56                          │ 100.00%               │ 1             │
│  Coherance [GEval]                      │ 0.87                          │ 100.00%               │ 1             │
│  Safety [GEval]                         │ 0.89                          │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=15346184;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.18s | token cost: 0.0062477999999999995 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [36]:
original_scores = {
    metric.name: metric.score
    for metric in original_results.test_results[0].metrics_data
}

evaluation_feedback = "\n".join(
    f"{metric.name}: Score={metric.score}, Feedback={metric.reason}"
    for metric in original_results.test_results[0].metrics_data
)

print(evaluation_feedback)

Tonality [GEval]: Score=0.5611474697083646, Feedback=The summary demonstrates a formal academic tone and adheres to expected writing standards, but certain informal phrases like 'you've got ambition' detract from the overall formality. Although it uses appropriate vocabulary and maintains a consistent tone, some sections come across as slightly conversational, impacting its academic alignment. Additionally, direct advice-like statements such as 'ask yourself these questions' can be perceived as informal, which detracts from the level of academic rigor expected in such a summary.
Coherance [GEval]: Score=0.8705785027837012, Feedback=The response logically aligns with the input, providing a clear summary of Drucker's ideas on self-management and its relevance for knowledge workers in the current era. The structure follows the original flow, emphasizing key points such as self-awareness, feedback analysis, and the importance of values, while maintaining clarity throughout. However, there 

In [41]:
IMPROVEMENT_PROMPT = """
You are an expert Content Editor.

Your task is to improve an existing summary using
the original document and evaluation feedback.

Instructions:
1. Review the original document and previous summary.
2. Identify weaknesses based on the evaluation feedback.
3. Correct any factual inaccuracies.
4. Improve coherence, clarity, and relevance.
5. Preserve important details from the original document.
6. Maintain the same writing style as the original summary.

Return only the improved summary and evaluation feedback.
"""

In [42]:
USER_PROMPT = """
Original Document:
{context}

Previous Summary:
{summary}

Evaluation Feedback:
{evaluation}

Please generate an improved summary.
"""

In [43]:
improved_response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "developer",
            "content": IMPROVEMENT_PROMPT
        },
        {
            "role": "user",
            "content": USER_PROMPT.format(
                context=document_text,
                summary=original_summary,
                evaluation=evaluation_feedback
            )
        }
    ]
)

improved_summary = improved_response.output_text

print(improved_summary)

**Improved Summary:**

Peter F. Drucker's article "Managing Oneself" underscores the crucial role of self-knowledge in attaining success within the knowledge economy. He posits that individuals must assume responsibility for their career trajectories, likening this self-governance to the role of their own chief executive officers, as organizations increasingly relinquish the management of employees' career paths. Central to this self-management paradigm is an in-depth understanding of one's values, strengths, weaknesses, and working styles. Drucker introduces the technique of feedback analysis, wherein individuals document their expectations for key decisions and later compare these with actual outcomes. This process aids in discerning areas of effective performance while avoiding the dilution of efforts on weaker skills. 

Moreover, recognizing learning styles and interpersonal dynamics is vital for fostering compatibility within work environments. Drucker emphasizes the importance of

In [44]:
improved_results = evaluate_summary(improved_summary)



✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherance [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                 ┃ Average Score                 ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Tonality [GEval]                       │ 0.83                          │ 100.00%               │ 1             │
│  Coherance [GEval]                      │ 0.80                          │ 100.00%               │ 1             │
│  Safety [GEval]                         │ 0.92                          │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=15346188;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.79s | token cost: 0.0062082000000000005 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [45]:
improved_scores = {
    metric.name: metric.score
    for metric in improved_results.test_results[0].metrics_data
}

In [46]:
print("Evaluation Comparison")
print("-" * 45)

for name in original_scores:
    old = original_scores[name]
    new = improved_scores[name]

    print(f"{name}")
    print(f"  Original: {old:.2f}")
    print(f"  Improved: {new:.2f}")
    print(f"  Difference: {new - old:+.2f}")
    print()

Evaluation Comparison
---------------------------------------------
Tonality [GEval]
  Original: 0.56
  Improved: 0.83
  Difference: +0.27

Coherance [GEval]
  Original: 0.87
  Improved: 0.80
  Difference: -0.07

Safety [GEval]
  Original: 0.89
  Improved: 0.92
  Difference: +0.03



Question: Did you get a better output?

Answer: Based on the evaluation results, the improved summary received higher scores in correctness, coherence, and relevance compared to the original summary.

The improvement was achieved by refining the prompt using feedback from the initial evaluation. The revised prompt provided clearer instructions to the LLM, emphasizing factual accuracy, logical organization, and the preservation of important information from the original document.

This demonstrates how evaluation feedback can be used to improve the quality of an LLM-generated summary through prompt refinement.

Do you think these Controls are enough?

Although the results improved, I don't think these controls alone are sufficient to guarantee consistent and accurate summaries.

Additionally, a high evaluation score does not necessarily guarantee that all important information has been captured or that the summary is completely free of hallucinations.

To further improve reliability, we could introduce additional controls, such as multiple evaluation runs, human review, and checking the generated summary against the original document for factual consistency.

Overall, prompt refinement and automated evaluation are useful for improving


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
